# Phase 1 / Stage 2 - Plant disease detection (MobileNetV2)

Capstone Project 7, Group 4. This notebook trains, evaluates and publishes the disease classifier
and hands it over to Phase 5.

**What it does**

1. Downloads PlantVillage (color) from Kaggle via `kagglehub`.
2. Fine-tunes `google/mobilenet_v2_1.0_224` (stage 1) and tags `v1.0`.
3. Reports PlantVillage accuracy, per-class F1, confusion matrix, top-3 and a threshold sweep.
4. **Legacy** section 7: the 50-image real-world set through the live-app chain (used to choose v1.0's threshold).
5. Pushes to **https://huggingface.co/neghani/plantvillage-mobilenetv2**.
6. **Stage 2 (section 7b):** continues from `v1.0` on a 50/50 mix of PlantVillage and Cropped-PlantDoc
   (Singh et al. 2020, CC BY 4.0; 27 of 38 classes). Diagnostics and calibration use **validation only**.
   The final test runs **once** per revision (guard on Hub/Drive). Publish `v1.1` only if PlantDoc val
   macro-F1 beats v1.0 and PlantVillage accuracy drops by at most 2 points.

All logic lives in `model/*.py`; this notebook only runs those scripts.

**How to test (local vs Colab)**

- **Local (CPU):** `python -m model.train --smoke` (synthetic data, ~2 min, writes only `model/outputs/smoke/`, never the final-test guard). Then `git push`.
- **Colab (GPU):** do **not** `files.upload()` code. Re-run the sync cell. It checks out a **pinned commit**, prints `CODE_COMMIT`, and leaves `model/outputs/` and images alone.
- **Real-world images:** git-ignored. Keep a zip on Drive (`--real-world-set-from`).

**Before you start (Colab)**

- Runtime -> Change runtime type -> **T4 GPU** (or better).
- Colab secrets: `HF_TOKEN` (write token). Optional: `KAGGLE_USERNAME` and `KAGGLE_KEY`.
- After you push stage-2 code, set `REPO_COMMIT` in the sync cell to that SHA so Colab cannot silently drift.

In [ ]:
!nvidia-smi -L || echo "No GPU - training will be very slow; switch the runtime to a GPU."

## 1. Sync the repository (pinned commit)

Fetches `notebooks/colab_sync.py` from the **same branch**, then hard-resets tracked files to
`REPO_COMMIT` (or `origin/REPO_BRANCH` if that is empty). Default branch is `Phase4-integration-path`
(Stage 2 lives there, not on `main`). Prints `CODE_COMMIT` for `run_info.json`.

After you push a local fix, **re-run this cell**. Do not upload `.py` files.

In [ ]:
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

# Stage 2 is on this branch. Pin REPO_COMMIT to a SHA after a push; empty -> branch tip.
REPO_BRANCH = os.environ.get("REPO_BRANCH", "Phase4-integration-path")
REPO_COMMIT = os.environ.get("REPO_COMMIT", "")

REPO = "neghani/iiith-group4-capstone-plat-leaf-decease-detection"
ref = REPO_COMMIT or REPO_BRANCH
RAW = f"https://raw.githubusercontent.com/{REPO}/{ref}/notebooks/colab_sync.py"
helper = Path("/tmp/colab_sync.py")
try:
    urllib.request.urlretrieve(RAW, helper)
    print("Fetched", RAW)
except Exception as exc:
    local = Path("notebooks/colab_sync.py")
    if not local.exists():
        raise RuntimeError("Could not fetch colab_sync.py from GitHub; clone the repo first.") from exc
    helper = local
    print("Using local", helper, "because raw fetch failed:", exc)

spec = importlib.util.spec_from_file_location("colab_sync", helper)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
argv = ["--branch", REPO_BRANCH]
if REPO_COMMIT:
    argv += ["--commit", REPO_COMMIT]
mod.main(argv)

root = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
os.chdir(root)
os.environ["REPO_ROOT"] = str(root)
os.environ["PYTHONPATH"] = str(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["CODE_COMMIT"] = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
print("cwd", os.getcwd())
print("CODE_COMMIT", os.environ["CODE_COMMIT"])
assert (root / "model" / "prepare_plantdoc.py").exists(), (
    f"{root}/model/prepare_plantdoc.py missing — section 1 checked out the wrong ref. "
    f"Use REPO_BRANCH=Phase4-integration-path (not main)."
)

Optional: replace **one** file after a local push (still from GitHub, always overwrite). Change the path and re-run. Do not `files.upload()`.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

run("notebooks/colab_sync.py", "--skip-install", "--branch", "Phase4-integration-path", "--file", "model/train.py")
# Extra files:
# run("notebooks/colab_sync.py", "--skip-install", "--branch", "Phase4-integration-path",
#     "--file", "model/evaluate.py", "--file", "model/predict.py", "--file", "model/common.py")


## 2. Log in to Hugging Face (and Kaggle if needed)

The push target is fixed to `neghani/plantvillage-mobilenetv2`. If you run this with a different account, set `HF_MODEL_ID` to your own namespace below. Tokens come from Colab secrets.

In [ ]:
import os
from huggingface_hub import HfApi, login

try:
    from google.colab import userdata  # type: ignore
    token = userdata.get("HF_TOKEN")
    for key in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        try:
            val = userdata.get(key)
        except Exception:
            val = None
        if val:
            os.environ[key] = val
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN to Colab secrets (enable 'Notebook access') or export HF_TOKEN."
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token

HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "neghani/plantvillage-mobilenetv2")
me = HfApi().whoami()["name"]
expected_owner = HF_MODEL_ID.split("/")[0]
assert me == expected_owner or expected_owner in [o["name"] for o in HfApi().whoami().get("orgs", [])], (
    f"Logged in as '{me}' but HF_MODEL_ID is '{HF_MODEL_ID}'. "
    f"Either log in as '{expected_owner}' or set os.environ['HF_MODEL_ID'] = '{me}/plantvillage-mobilenetv2' before this cell."
)
os.environ["HF_MODEL_ID"] = HF_MODEL_ID
print("Logged in as", me, "-> will push to", HF_MODEL_ID)
print("Kaggle secrets:", "yes" if os.environ.get("KAGGLE_USERNAME") else "not set (public dataset usually works anyway)")

## 3. Download PlantVillage (color)

In [ ]:
import kagglehub, os
root = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print("Downloaded to", root)

import sys
sys.path.insert(0, "model")
from common import resolve_data_dir
DATA_DIR = str(resolve_data_dir(root))
os.environ["DATA_DIR"] = DATA_DIR
classes = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
print(len(classes), "classes, e.g.", classes[:5])
print("images:", sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes))

## 4. Quick smoke test (optional)

**Local (no downloads):** `python -m model.train --smoke` — synthetic images, 1–2 CPU steps, writes only
`model/outputs/smoke/`, never the final-test guard.

**Colab (optional, ~2 min):** tomato classes only, 60 images per class, 1 epoch on real PlantVillage.
Writes `model/outputs/tomato_smoke/` so it does not collide with the synthetic smoke dir. Nothing is pushed.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

require_env("DATA_DIR")
run("model/train.py",
    "--data_dir", os.environ["DATA_DIR"], "--crop_filter", "Tomato", "--max_samples_per_class", "60",
    "--output_dir", "model/outputs/tomato_smoke",
    "--do_train", "--do_eval", "--num_train_epochs", "1",
    "--per_device_train_batch_size", "32", "--per_device_eval_batch_size", "64",
    "--learning_rate", "5e-4", "--fp16", "--report_to", "none", "--save_strategy", "no", "--logging_steps", "5")
run("model/evaluate.py", "--model", "model/outputs/tomato_smoke")


## 5. Full training run

All 38 classes (~54k images). Checkpoints go to **Drive** (`MyDrive/plant-disease/stage1/...`) so a
disconnect does not wipe the run. Re-running this cell **resumes**, or **skips** if the run already
finished. Pass `--overwrite_output_dir` only to throw it away and start over.


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

require_env("DATA_DIR", "HF_MODEL_ID")
drive = Path("/content/drive/MyDrive/plant-disease")
OUTPUT_DIR = str(drive / "stage1/plantvillage-mobilenetv2") if drive.exists() else "model/outputs/plantvillage-mobilenetv2"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
print("OUTPUT_DIR", OUTPUT_DIR)

run("model/train.py",
    "--model_name_or_path", "google/mobilenet_v2_1.0_224",
    "--data_dir", os.environ["DATA_DIR"],
    "--output_dir", OUTPUT_DIR,
    "--resume_from_checkpoint", "auto",
    "--do_train", "--do_eval",
    "--num_train_epochs", "5",
    "--per_device_train_batch_size", "64", "--per_device_eval_batch_size", "128",
    "--learning_rate", "5e-4", "--weight_decay", "0.01", "--warmup_steps", "0.05",
    "--lr_scheduler_type", "cosine",
    "--fp16", "--dataloader_num_workers", "2",
    "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
    "--load_best_model_at_end", "--metric_for_best_model", "macro_f1", "--greater_is_better", "True",
    "--logging_steps", "50", "--report_to", "none", "--seed", "42",
    "--push_to_hub", "--hub_model_id", os.environ["HF_MODEL_ID"], "--hub_strategy", "end")


## 6. Evaluate on the held-out test split and build the model card

Produces `reports/` (metrics.json, classification report, confusion matrix, threshold sweep) and fills `README.md`; `--push` uploads both to the Hub.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

require_env("OUTPUT_DIR", "HF_MODEL_ID")
run("model/evaluate.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--push")


In [ ]:
from IPython.display import Image, display
import json, os
display(Image(filename=os.path.join(os.environ["OUTPUT_DIR"], "reports", "confusion_matrix.png"), width=900))
m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy", "recommended_confidence_threshold")})

## 7. Real-world test set (legacy v1.0 path)

Used to choose crop strategy B and threshold 0.85 for `v1.0`. Stage 2 (section 7b) does **not**
retune on this set: the 50 images are in the final test only, excluded from training, and the
threshold is chosen on PlantDoc **validation**. Keep this section for reproducing the v1.0 card.

The labelled set lives in `model/data/real_world_set/` (`labels.csv` with a `tune` / `final` split,
`sources.csv`, optional `card_notes.md`, and `images/`). The GitHub repo is public, so the images are
git-ignored and come from Drive: `MyDrive/plant-disease/real_world_set.zip` (zip the whole folder).

**New or changed set:** replace that zip on Drive and re-run this section. The next cell wipes the
local set and copies the zip in. The eval cell then runs the live-app chain (API quality gate + leaf
crop -> model) on `tune` and `final`, and compares a fingerprint of `labels.csv` + every image, the
model weights and the threshold with `reports/real_world_set/<split>/run.json` on the Hub:

- something changed (or the card was overwritten) -> results are uploaded and the **Hub README's
  real-world section and `model-index` are regenerated** from the `final` split;
- nothing changed -> it prints `unchanged` and pushes nothing.

Set `FORCE_PUBLISH = True` to republish anyway.

In [ ]:
# images/ is git-ignored. Prefer the Drive zip; otherwise rebuild from Cropped-PlantDoc (seed 42).
import os, subprocess, sys
from pathlib import Path

root = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
os.chdir(root)
REAL_WORLD_SET_SRC = Path("/content/drive/MyDrive/plant-disease/real_world_set.zip")
dest = root / "model" / "data" / "real_world_set"
plantdoc = root / "model" / "data" / "plantdoc_src" / "train"

if REAL_WORLD_SET_SRC.exists():
    subprocess.check_call([sys.executable, "notebooks/colab_sync.py", "--skip-install",
                           "--real-world-set-from", str(REAL_WORLD_SET_SRC)], cwd=root)
elif plantdoc.is_dir():
    print("No Drive zip; rebuilding the 50-image set from", plantdoc.parent)
    subprocess.check_call([sys.executable, "model/build_real_world_set.py", str(plantdoc.parent), str(dest)], cwd=root)
else:
    raise SystemExit(
        f"No images at {dest}/images and no zip at {REAL_WORLD_SET_SRC}.\n"
        "Put real_world_set.zip on Drive, or run section 7b prep first (clones PlantDoc)."
    )
n_img = sum(1 for p in dest.joinpath("images").rglob("*") if p.is_file())
print(f"real_world_set images: {n_img}")
assert n_img >= 50, f"expected 50 images under {dest}/images, got {n_img}"

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

import json, os
from pathlib import Path

FORCE_PUBLISH = False

# Model under test: this session's run if it exists, else the published release.
out_dir = Path(os.environ.get("OUTPUT_DIR", "model/outputs/plantvillage-mobilenetv2"))
if (out_dir / "model.safetensors").exists():
    EVAL_MODEL, EVAL_REV = str(out_dir), ""
else:
    EVAL_MODEL, EVAL_REV = os.environ["HF_MODEL_ID"], os.environ.get("HF_MODEL_REVISION", "v1.0")
# Crop strategy + threshold chosen on the tune split by model/eval_strategies.py (= api/app/config.py).
STRATEGY, THRESHOLD = "B", 0.85
print(f"Testing {EVAL_MODEL}{'@' + EVAL_REV if EVAL_REV else ''} with strategy {STRATEGY} at threshold {THRESHOLD}")

rev_flag = f"--revision {EVAL_REV}" if EVAL_REV else ""
change_flag = "" if FORCE_PUBLISH else "--only_if_changed"
# tune first (thresholds are chosen there), final last: only final rewrites the Hub README.
require_env("HF_MODEL_ID")
for split in ("tune", "final"):
    args = [
        "model/eval_real_world_set.py",
        "--set_dir", "model/data/real_world_set", "--split", split,
        "--predictor", "app_predictor:diagnose", "--strategy", STRATEGY, "--threshold", str(THRESHOLD),
        "--model", EVAL_MODEL, "--hub_model_id", os.environ["HF_MODEL_ID"],
        "--out", f"model/outputs/real_world/{split}", "--push",
    ]
    if EVAL_REV:
        args += ["--revision", EVAL_REV]
    if not FORCE_PUBLISH:
        args.append("--only_if_changed")
    run(*args)


In [ ]:
from IPython.display import Markdown, display
report = Path("model/outputs/real_world/final/report.md")
display(Markdown(report.read_text()) if report.exists() else Markdown("_No final report yet._"))
print(f"Hub card: https://huggingface.co/{os.environ['HF_MODEL_ID']}")

### Optional: ad-hoc photo folder (not published to the card)

Loose photos in `model/data/real_world/<ClassName>/` from `MyDrive/plant-disease/real_world.zip`,
scored full-frame by `model/real_world_eval.py`. Kept for quick checks; the card's real-world section
comes from the labelled set above.

In [ ]:
# REAL_WORLD_SRC = "/content/drive/MyDrive/plant-disease/real_world.zip"
# !python notebooks/colab_sync.py --skip-install --real-world-from $REAL_WORLD_SRC
# !python model/real_world_eval.py --model "$OUTPUT_DIR" --hub_model_id "$HF_MODEL_ID"

## 7b. Stage 2: fine-tune on Cropped-PlantDoc, calibrate, test once

`v1.0` was trained on PlantVillage only (one leaf, plain background). This stage continues from those
weights on a **50/50 mix** of PlantVillage and [Cropped-PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset)
(Singh et al. 2020, CC BY 4.0), pinned at `5467f6012d78d1c446145d5f582da6096f852ae8`. 27 of 38 PlantVillage
classes appear in PlantDoc; the other 11 stay lab-only. Spider-mites is dropped.

**Rules**

- Decisions use **PlantDoc validation only**. Do not look at the final test until `final_test.py`.
- Report sample sizes next to every percentage.
- If the final pass rate is still below 50%, report it honestly. Do not change the threshold or the data after seeing it.
- Checkpoints go to Drive every epoch (`--resume_from_checkpoint auto`). Small artifacts upload to Hub `stage2-runs` and Drive after each step.
- The final-test **guard** is `hub:$HF_MODEL_ID` (JSON named by revision on `stage2-runs`) or `drive:.../final_test_guard`.

**Needs this session:** section 1 (pinned `CODE_COMMIT`), section 2 (HF token), section 3 (`DATA_DIR`),
section 7's real-world set (excluded from train by `original_path`).

In [ ]:
import os, sys, subprocess
from pathlib import Path

ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "prepare_plantdoc.py").exists():
    raise SystemExit(
        f"Stage 2 code is not at {ROOT}. Re-run section 1 with REPO_BRANCH=Phase4-integration-path "
        "(section 1 was checking out main, which does not have python -m model.*)."
    )
os.chdir(ROOT)
os.environ["REPO_ROOT"] = str(ROOT)
os.environ["PYTHONPATH"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["CODE_COMMIT"] = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("cwd", os.getcwd())
print("CODE_COMMIT", os.environ["CODE_COMMIT"])
if not os.environ.get("DATA_DIR"):
    print("NOTE: DATA_DIR is unset. Re-run section 3 before the train cells (prep can still run).")

DRIVE = "/content/drive/MyDrive/plant-disease"
STAGE2 = f"{DRIVE}/stage2"
RUN = "stage2"
ARTIFACTS = "model/outputs"
os.makedirs(STAGE2, exist_ok=True)

def run_mod(module, *args):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, "-m", module, *map(str, args)]
    print("$", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=ROOT, env=env)


def apply_protocol_hotfix():
    """PlantVillage vs PlantDoc-test Hamming-6 across crops is a hash collision, not leakage."""
    proto = ROOT / "model" / "data_protocol.py"
    trainp = ROOT / "model" / "train.py"
    pt, tt = proto.read_text(), trainp.read_text()
    if "MATCH_DIST_CROSS" in pt and 'label=Path(rel).parts[0]' in tt:
        print("protocol hotfix already present")
        return
    old_m = "MATCH_DIST = 6  # pHash Hamming distance at or below which two images are near-duplicates"
    if old_m in pt:
        pt = pt.replace(old_m, old_m.replace("two images", "two PlantDoc images") + "\nMATCH_DIST_CROSS = 3", 1)
    old_pairs = '''def _pairs_across(a: list[Item], b: list[Item], max_dist: int = MATCH_DIST) -> list[str]:
    problems = []
    paths_a = {norm_path(x.path) for x in a}
    for x in b:
        if norm_path(x.path) in paths_a:
            problems.append(f"shared path {x.path}")
    ha = {x.path: x.phash for x in a}
    hb = {x.path: x.phash for x in b}
    for qa, qb, d in near_dups(ha, hb, max_dist):
        problems.append(f"pHash dist {d}: {qa} ~ {qb}")
    return problems'''
    new_pairs = '''def crop_prefix(name: str) -> str:
    return (name or "").split("___")[0]


def _pairs_across(a: list[Item], b: list[Item], max_dist: int = MATCH_DIST,
                  same_crop: bool = False) -> list[str]:
    problems = []
    paths_a = {norm_path(x.path) for x in a}
    by_path_a = {x.path: x for x in a}
    by_path_b = {x.path: x for x in b}
    for x in b:
        if norm_path(x.path) in paths_a:
            problems.append(f"shared path {x.path}")
    ha = {x.path: x.phash for x in a}
    hb = {x.path: x.phash for x in b}
    for qa, qb, d in near_dups(ha, hb, max_dist):
        if same_crop:
            ca, cb = crop_prefix(by_path_a[qa].label), crop_prefix(by_path_b[qb].label)
            if ca and cb and ca != cb:
                continue
        problems.append(f"pHash dist {d}: {qa} ~ {qb}")
    return problems'''
    if old_pairs in pt:
        pt = pt.replace(old_pairs, new_pairs, 1)
    old_extra = '        problems += [f"extra (PlantVillage) vs test: {p}" for p in _pairs_across(extra, test)]'
    new_extra = ('        problems += [f"extra (PlantVillage) vs test: {p}" for p in _pairs_across(\n'
                 '            extra, test, max_dist=MATCH_DIST_CROSS, same_crop=True)]')
    if old_extra in pt:
        pt = pt.replace(old_extra, new_extra, 1)
    old_item = ('        extra_items.append(Item(path=str(path), split="train", label="", phash=phash_cached(path, cache),\n'
                '                                group=f"pv:{rel}", source="plantvillage"))')
    new_item = ('        extra_items.append(Item(path=str(path), split="train", label=Path(rel).parts[0],\n'
                '                                phash=phash_cached(path, cache),\n'
                '                                group=f"pv:{rel}", source="plantvillage"))')
    if old_item in tt:
        tt = tt.replace(old_item, new_item, 1)
    proto.write_text(pt)
    trainp.write_text(tt)
    if "MATCH_DIST_CROSS" not in proto.read_text():
        raise SystemExit("protocol hotfix failed; paste the Colab patch cell from chat")
    print("protocol hotfix applied (PlantVillage vs test: same-crop, dist<=3)")


apply_protocol_hotfix()

def upload(step, *paths):
    missing = [p for p in paths if not Path(p).exists()]
    if missing:
        raise SystemExit(f"upload {step}: missing {missing} (the previous step did not finish)")
    run_mod("model.artifacts", "--step", step, "--run", RUN,
            "--hub", os.environ["HF_MODEL_ID"], "--drive", f"{STAGE2}/artifacts", *paths)

# Clones Cropped-PlantDoc if needed, excludes real_world_set + pHash dups + spider-mites, asserts protocol.
run_mod("model.prepare_plantdoc", "--exclude_set", "model/data/real_world_set", "--artifacts_dir", ARTIFACTS, "--skip_if_exists")
upload("prep", "model/outputs/data_manifest.json", "model/data/plantdoc_train/prep_report.md",
       "model/data/plantdoc_train/prep_report.json", "model/data/plantdoc_train/excluded.csv")


In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
display(Markdown(Path("model/data/plantdoc_train/prep_report.md").read_text()))

### Diagnostics on PlantDoc validation (v1.0, before fine-tuning)

Validation only. Writes `diagnostics_v1_0.json` used later by the publish gate. Crop masking is off.

In [ ]:
run_mod("model.diagnostics", "--model", os.environ["HF_MODEL_ID"], "--revision", "v1.0",
        "--manifest", "model/data/plantdoc_train/manifest.csv",
        "--extra_dir", "model/data/plantdoc_train", "--out", "model/outputs")
upload("diagnostics", "model/outputs/diagnostics_v1_0.json", "model/outputs/diagnostics_v1_0.md")

### Train from `v1.0` (two learning rates, checkpoints on Drive)

Mixed sampler: 50% PlantDoc (class weight `1/sqrt(freq)`), 50% PlantVillage, 4000 draws/epoch, 8 epochs,
label smoothing 0.1, cosine schedule, PlantDoc RRC (0.5, 1.0). Best checkpoint by **PlantDoc validation
macro-F1**. Resume with `--resume_from_checkpoint auto` after a disconnect. Compare 1e-4 and 5e-5 on
validation only, then keep the winner.

In [ ]:
LR1 = f"{STAGE2}/lr1e-4"
run_mod(
    "model.train", "--init_from", os.environ["HF_MODEL_ID"], "--model_revision", "v1.0",
    "--data_dir", os.environ["DATA_DIR"], "--extra_data", "plantdoc",
    "--plantdoc_fraction", "0.5", "--epoch_samples", "4000", "--artifacts_dir", "model/outputs",
    "--output_dir", LR1, "--resume_from_checkpoint", "auto",
    "--do_train", "--do_eval", "--num_train_epochs", "8",
    "--per_device_train_batch_size", "64", "--per_device_eval_batch_size", "128",
    "--learning_rate", "1e-4", "--weight_decay", "0.01", "--label_smoothing_factor", "0.1",
    "--lr_scheduler_type", "cosine", "--fp16", "--dataloader_num_workers", "2",
    "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
    "--load_best_model_at_end", "--metric_for_best_model", "plantdoc_macro_f1",
    "--greater_is_better", "True", "--logging_steps", "50", "--report_to", "none", "--seed", "42",
)
upload("train_lr1e-4", f"{LR1}/run_info.json", f"{LR1}/eval_results.json")

Second learning rate (5e-5). Same protocol; separate Drive folder so the first run is not overwritten.

In [ ]:
LR2 = f"{STAGE2}/lr5e-5"
run_mod(
    "model.train", "--init_from", os.environ["HF_MODEL_ID"], "--model_revision", "v1.0",
    "--data_dir", os.environ["DATA_DIR"], "--extra_data", "plantdoc",
    "--plantdoc_fraction", "0.5", "--epoch_samples", "4000", "--artifacts_dir", "model/outputs",
    "--output_dir", LR2, "--resume_from_checkpoint", "auto",
    "--do_train", "--do_eval", "--num_train_epochs", "8",
    "--per_device_train_batch_size", "64", "--per_device_eval_batch_size", "128",
    "--learning_rate", "5e-5", "--weight_decay", "0.01", "--label_smoothing_factor", "0.1",
    "--lr_scheduler_type", "cosine", "--fp16", "--dataloader_num_workers", "2",
    "--eval_strategy", "epoch", "--save_strategy", "epoch", "--save_total_limit", "2",
    "--load_best_model_at_end", "--metric_for_best_model", "plantdoc_macro_f1",
    "--greater_is_better", "True", "--logging_steps", "50", "--report_to", "none", "--seed", "42",
)
upload("train_lr5e-5", f"{LR2}/run_info.json", f"{LR2}/eval_results.json")

In [ ]:
import json, os
from pathlib import Path

def plantdoc_f1(run):
    p = Path(run) / "eval_results.json"
    if not p.exists():
        p = Path(run) / "all_results.json"
    ev = json.loads(p.read_text())
    return float(ev.get("eval_plantdoc_macro_f1") or ev.get("eval_macro_f1") or 0), ev

f1_a, ev_a = plantdoc_f1(LR1)
f1_b, ev_b = plantdoc_f1(LR2)
print("1e-4 PlantDoc val macro-F1", f1_a)
print("5e-5 PlantDoc val macro-F1", f1_b)
BEST = LR1 if f1_a >= f1_b else LR2
os.environ["STAGE2_DIR"] = BEST
print("keeping", BEST)

run_mod("model.evaluate", "--model", BEST, "--data_dir", os.environ["DATA_DIR"],
        "--reports_dir", f"{BEST}/reports", "--card_revision", "v1.1")
m = json.load(open(os.path.join(BEST, "reports", "metrics.json")))
print({k: m[k] for k in ("accuracy", "macro_f1", "top3_accuracy")}, "(v1.0: 0.9912, 0.9858, 0.9994)")
upload("evaluate", f"{BEST}/reports/metrics.json", f"{BEST}/run_info.json")

### Calibrate on PlantDoc validation

Fits temperature T by NLL on validation logits. Sweeps the live chain (strategy B) 0.30–0.95;
keeps the highest pass rate among thresholds with ≥70% precision of 'ok' answers. If none qualify,
the threshold is **provisional** and the API config is left unchanged.

In [ ]:
run_mod("model.calibrate", "--model", os.environ["STAGE2_DIR"],
        "--manifest", "model/data/plantdoc_train/manifest.csv",
        "--extra_dir", "model/data/plantdoc_train")
upload("calibrate", f"{os.environ['STAGE2_DIR']}/calibration.json",
       f"{os.environ['STAGE2_DIR']}/threshold.json",
       f"{os.environ['STAGE2_DIR']}/calibration_report.md")

### Final test (once per revision) then publish gate

PlantDoc `test/` plus the 50-image set, frozen T and threshold. Guard: `hub:$HF_MODEL_ID` on the
`stage2-runs` branch (`final_test_guard/v1.1.json`). A second run is refused unless `--force`.
Do not change the threshold or the data after seeing this number.

Publish `v1.1` only if PlantDoc val macro-F1 beats v1.0 **and** PlantVillage accuracy drops by at most 2 points. The API stays on `v1.0` until `HF_MODEL_REVISION` is changed.

In [ ]:
run_mod("model.final_test", "--v11", os.environ["STAGE2_DIR"], "--revision_name", "v1.1",
        "--guard", f"hub:{os.environ['HF_MODEL_ID']}",
        "--v10", os.environ["HF_MODEL_ID"], "--v10_revision", "v1.0",
        "--test_dir", "model/data/plantdoc_test", "--real_world", "model/data/real_world_set",
        "--out", "model/outputs")
upload("final_test", "model/outputs/final_report_v1_1.md", "model/outputs/final_test_per_image.csv")

run_mod("model.publish", "--model", os.environ["STAGE2_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"],
        "--tag", "v1.1", "--diagnostics", "model/outputs/diagnostics_v1_0.json", "--dry_run")
# Uncomment only after the dry run and the gate both pass:
# run_mod("model.publish", "--model", os.environ["STAGE2_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"],
#         "--tag", "v1.1", "--diagnostics", "model/outputs/diagnostics_v1_0.json",
#         "--tag_message", "Stage 2: PlantVillage + Cropped-PlantDoc",
#         "--commit_message", "Stage 2: fine-tune on PlantDoc")

## 8. Publish and tag the release for Phase 5

`model/publish.py` uses the `HF_TOKEN` Colab secret: it checks the token and owner, uploads the final model folder
(weights, `config.json`, `splits.json`, `reports/`, model card; no checkpoints) and tags `v1.0`.
Run the dry run first if you want to see the file list.

Phase 5 pins `HF_MODEL_REVISION=v1.0` so later retraining cannot silently change the API's behaviour.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

require_env("OUTPUT_DIR", "HF_MODEL_ID")
run("model/publish.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--dry_run")


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("REPO_ROOT") or "/content/iiith-group4-capstone-plat-leaf-decease-detection")
if not (ROOT / "model" / "train.py").exists():
    raise SystemExit(f"Repo not at {ROOT}. Re-run section 1 (REPO_BRANCH=Phase4-integration-path).")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ["REPO_ROOT"] = str(ROOT)
os.environ.setdefault("PYTHONPATH", str(ROOT))

def run(*argv):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(ROOT)
    cmd = [sys.executable, *map(str, argv)]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=ROOT, env=env)

def require_env(*names):
    missing = [n for n in names if not os.environ.get(n)]
    if missing:
        raise SystemExit(f"Missing env {missing}. Re-run the cells that set them (section 2/3).")

require_env("OUTPUT_DIR", "HF_MODEL_ID")
run("model/publish.py", "--model", os.environ["OUTPUT_DIR"], "--hub_model_id", os.environ["HF_MODEL_ID"], "--tag", "v1.0")


## 9. Inference demo from the Hub (what Phase 5 will do)

In [ ]:
import json, os, random
from PIL import Image as PILImage
from IPython.display import display

import sys
sys.path.insert(0, "model")
from predict import diagnose

m = json.load(open(os.path.join(os.environ["OUTPUT_DIR"], "reports", "metrics.json")))
threshold = m["recommended_confidence_threshold"]

data_dir = os.environ["DATA_DIR"]
cls = random.choice(sorted(os.listdir(data_dir)))
img_path = os.path.join(data_dir, cls, random.choice(os.listdir(os.path.join(data_dir, cls))))
display(PILImage.open(img_path).resize((224, 224)))
print("True class:", cls)
print(json.dumps(diagnose(img_path, model_id=os.environ["HF_MODEL_ID"], revision="v1.0", threshold=threshold), indent=2))

## 10. Handover to Phase 5

Put these in `.env` of the API service:

```env
HF_MODEL_ID=neghani/plantvillage-mobilenetv2
HF_MODEL_REVISION=v1.0
CONFIDENCE_THRESHOLD=<recommended_confidence_threshold from reports/metrics.json>
```

- `model/predict.py` is the reference implementation (quality gate -> model -> top-k -> not sure).
- Class names in `config.json` (`id2label`) are PlantVillage folder names and match the keys of
  `api/data/knowledge/diseases.json`.
- Full instructions, challenges and the ResNet fallback are in `model/README.md`.